# MTPL2 Frequency Modelling Walkthrough

End-to-end Poisson frequency model on the French MTPL2 dataset (678k
policies). Covers data prep, model fitting, relativity inspection,
and the impact of discretisation on speed and accuracy.

Sections:

1. **Data loading & preparation**
2. **Model specification** — splines, categoricals, numerics
3. **Fitting with REML** — automatic smoothness selection
4. **Relativity plots** — grid, individual terms, layout options
5. **Summary table** — statsmodels-style output
6. **Discretisation** — `discrete=True` for fast fitting
7. **Continuous vs discretised** — side-by-side comparison

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from superglm import Categorical, Numeric, Poisson, Spline, SuperGLM

## 1. Data loading & preparation

The French MTPL2 frequency dataset contains 678k motor third-party
liability policies with claim counts and exposure. We fetch it from
OpenML (no local files needed) and apply standard actuarial cleaning.

In [ ]:
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame

# Standard cleaning
df["ClaimNb"] = df["ClaimNb"].astype(float).clip(upper=4)
df["Exposure"] = df["Exposure"].astype(float).clip(lower=0.01)
df["DrivAge"] = df["DrivAge"].astype(float).clip(18, 90)
df["VehAge"] = df["VehAge"].astype(float).clip(0, 20)
df["BonusMalus"] = df["BonusMalus"].astype(float).clip(50, 150)
df["Density"] = df["Density"].astype(float)
df["LogDensity"] = np.log1p(df["Density"])

y = df["ClaimNb"].values
exposure = df["Exposure"].values

features_to_use = ["DrivAge", "VehAge", "BonusMalus", "LogDensity", "Area", "VehGas"]
X = df[features_to_use]

print(f"Rows: {len(df):,}")
print(f"Claim rate: {y.sum() / exposure.sum():.4f}")
print(f"Zero fraction: {(y == 0).mean():.1%}")
X.describe().round(2)

## 2. Model specification

A typical actuarial frequency model mixes feature types:

| Feature | Type | Why |
|---------|------|-----|
| DrivAge | `Spline(kind="bs", n_knots=15)` | Smooth U-shaped effect |
| VehAge | `Spline(kind="bs", n_knots=10)` | Smooth decreasing effect |
| BonusMalus | `Spline(kind="bs", n_knots=8)` | Strongly monotone, needs flexibility |
| LogDensity | `Numeric()` | Log-linear in population density |
| Area | `Categorical()` | 6 unordered regions |
| VehGas | `Categorical()` | Diesel vs Regular |

In [ ]:
model = SuperGLM(
    family=Poisson(),
    discrete=True,
    features={
        "DrivAge": Spline(kind="cr", n_knots=15, knot_strategy="quantile_rows", penalty="ssp"),
        "VehAge": Spline(kind="cr", n_knots=10, knot_strategy="uniform", penalty="ssp"),
        "BonusMalus": Spline(
            kind="cr", n_knots=8, knot_strategy="quantile_tempered", knot_alpha=0.2, penalty="ssp"
        ),
        "LogDensity": Numeric(),
        "Area": Categorical(base="most_exposed"),
        "VehGas": Categorical(base="most_exposed"),
    },
)
print("Model specified.")

## 3. Fitting with REML

`fit_reml()` estimates smoothing parameters automatically via
restricted marginal likelihood (Wood 2011). No manual tuning of
lambda needed — REML finds the right amount of smoothing for
each spline.

In [ ]:
t0 = time.perf_counter()
model.fit_reml(X, y, exposure=exposure)
elapsed = time.perf_counter() - t0

print(f"Fit time: {elapsed:.1f}s")
print(f"REML converged: {model._reml_result.converged}")
print(f"REML iterations: {model._reml_result.n_reml_iter}")
print("\nPer-feature edf:")
for name in model._feature_order:
    ti = model.term_inference(name, with_se=False)
    edf_str = f"{ti.edf:.1f}" if ti.edf is not None else "—"
    print(f"  {name:15s} edf = {edf_str}")

## 4. Relativity plots

### 4a. All features in a grid

The default `model.plot()` shows all main effects in a 2-column grid
with exposure density overlays and pointwise CIs.

In [ ]:
fig = model.plot(X=X, sample_weight=exposure)
plt.show()

### 4b. Single term — large format

Plot one term at a time for detailed inspection. Useful for
presentations or reports where you want to focus on one effect.

In [ ]:
fig = model.plot(
    "DrivAge",
    X=X,
    sample_weight=exposure,
    show_knots=True,
    figsize=(8, 6),
    title="Driver Age Effect",
    subtitle="Poisson frequency model, REML smoothing",
)
plt.show()

### 4c. Subset of terms

Pass a list of feature names to plot a subset in a grid.

In [ ]:
fig = model.plot(
    ["DrivAge", "VehAge", "BonusMalus"],
    X=X,
    sample_weight=exposure,
    ncols=3,
    figsize=(16, 5),
    title="Smooth terms",
)
plt.show()

### 4d. Simultaneous confidence bands

Pointwise CIs are narrow but don't account for multiple testing across
the curve. Simultaneous bands (Wood 2006, posterior simulation) give
honest coverage for the entire curve.

In [ ]:
fig = model.plot(
    "DrivAge",
    ci="both",
    X=X,
    sample_weight=exposure,
    figsize=(10, 5),
    title="Pointwise vs Simultaneous Bands",
)
plt.show()

## 5. Summary table

statsmodels-style summary with coefficient estimates, smooth term
edf, and Wood (2013) p-values for smooth terms.

In [ ]:
model.summary()

## 6. Spline basis discretisation

With `discrete=True`, superglm bins the covariate values into a compact
grid and evaluates the spline basis on the unique bin centres rather
than on every observation. The gram matrix `B'WB` and matrix-vector
products then aggregate by bin first — `O(n_bins)` matrix ops + `O(n)`
scatter/gather — instead of `O(n)` dense operations.

This is the same idea as mgcv's `bam()`: discretised bases make
large-data GAM fitting practical without changing the statistical model.

In [ ]:
model_disc = SuperGLM(
    family=Poisson(),
    discrete=True,
    features={
        "DrivAge": Spline(kind="bs", n_knots=15, penalty="ssp"),
        "VehAge": Spline(kind="bs", n_knots=10, penalty="ssp"),
        "BonusMalus": Spline(kind="bs", n_knots=8, penalty="ssp"),
        "LogDensity": Numeric(),
        "Area": Categorical(base="most_exposed"),
        "VehGas": Categorical(base="most_exposed"),
    },
)

t0 = time.perf_counter()
model_disc.fit_reml(X, y, exposure=exposure)
elapsed_disc = time.perf_counter() - t0

print(f"Continuous fit: {elapsed:.1f}s")
print(f"Discretised fit: {elapsed_disc:.1f}s")
print(f"Speedup: {elapsed / elapsed_disc:.1f}x")
print(f"\nREML converged: {model_disc._reml_result.converged}")

## 7. Continuous vs discretised comparison

How much does discretisation change the fitted curves? We overlay
the relativities from both models to check.

In [ ]:
# Compare the three spline terms
spline_names = ["DrivAge", "VehAge", "BonusMalus"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, name in zip(axes, spline_names):
    ti_cont = model.term_inference(name)
    ti_disc = model_disc.term_inference(name)

    ax.plot(
        ti_cont.x,
        ti_cont.relativity,
        color="#1f77b4",
        linewidth=2,
        label=f"Continuous (edf={ti_cont.edf:.1f})",
    )
    ax.plot(
        ti_disc.x,
        ti_disc.relativity,
        color="#d62728",
        linewidth=2,
        linestyle="--",
        label=f"Discretised (edf={ti_disc.edf:.1f})",
    )
    ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("Relativity")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.2, axis="y")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    f"Continuous ({elapsed:.1f}s) vs Discretised ({elapsed_disc:.1f}s)",
    fontweight="bold",
    fontsize=13,
)
fig.tight_layout()
plt.show()

In [ ]:
# Numerical comparison: max absolute difference in relativities
print("Max absolute relativity difference (continuous vs discretised):")
for name in spline_names:
    ti_c = model.term_inference(name)
    ti_d = model_disc.term_inference(name)
    diff = np.max(np.abs(ti_c.relativity - ti_d.relativity))
    print(f"  {name:15s} {diff:.6f}")

print("\nMax edf difference:")
for name in spline_names:
    ti_c = model.term_inference(name)
    ti_d = model_disc.term_inference(name)
    print(f"  {name:15s} {abs(ti_c.edf - ti_d.edf):.3f}")

## 8. Key takeaways

| | Continuous | Discretised |
|---|---|---|
| **Fit time** | Baseline | Typically 2–5x faster |
| **Accuracy** | Exact | Near-identical (< 0.01 relativity diff) |
| **REML** | Full | Full (same convergence) |
| **When to use** | Small–medium data, final model | Large data, iteration, model selection |

The workflow: iterate with `discrete=True` for speed, then optionally
refit without discretisation for the final production model. In
practice the difference is negligible.